IMPORT THƯ VIỆN

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import os
import warnings
import pyodbc
import sqlalchemy
from sqlalchemy import create_engine, text

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)


ĐỌC DỮ LIỆU TỪ SQL

In [2]:
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=localhost;"  
    "DATABASE=NovaBank_CreditRisk;"
    "Trusted_Connection=yes;"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

query = "SELECT * FROM dbo.vw_credit_risk_base"
df_raw = pd.read_sql(query, conn)

df_bi = df_raw.copy()
df_ml = df_raw.copy()

## POWER BI
XỬ LÝ MISSING VALUES

In [ ]:
# Tạo missing flag MNAR
df_bi["emp_length_missing"] = df_bi["person_emp_length"].isnull().astype(int)

#Impute bằng median
median_emp = df_bi["person_emp_length"].median()
median_rate = df_bi["loan_int_rate"].median()

df_bi["person_emp_length"] = df_bi["person_emp_length"].fillna(median_emp)
df_bi["loan_int_rate"]  = df_bi["loan_int_rate"].fillna(median_rate)

#check lại dữ liệu sau khi impute 
print(df_bi[["person_emp_length", "loan_int_rate"]].isnull().sum())

person_emp_length    0
loan_int_rate        0
dtype: int64


XỬ LÝ OUTLIER

In [4]:
#ngưỡng tối đa cho từng biến
age_cap = 80
emp_cap = 60
dti_cap = 1.0

# Cap person_age
line_age = (df_bi["person_age"] > age_cap).sum()
df_bi["person_age"] = df_bi["person_age"].clip(upper=age_cap)

# Cap person_emp_length
line_emp = (df_bi["person_emp_length"] > emp_cap).sum()
df_bi["person_emp_length"] = df_bi["person_emp_length"].clip(upper=emp_cap)

# Cap debt_to_income_ratio
line_dti = (df_bi["debt_to_income_ratio"] > dti_cap).sum()
df_bi["debt_to_income_ratio"] = df_bi["debt_to_income_ratio"].clip(upper=dti_cap)


Tạo band 

In [ ]:

#Income band 
df_bi["income_band"] = pd.cut(
    df_bi["person_income"],
    bins=[0, 30000, 60000, 100000, 200000, np.inf],
    labels=["< 30K", "30-60K", "60-100K", "100-200K", "200K+"],
    include_lowest=True
)

# Age band 
df_bi["age_band"] = pd.cut(
    df_bi["person_age"],
    bins=[17, 23, 30, 36, 40, 50, 60, np.inf],
    labels=["18-23", "24-30", "31-36", "37-40", "41-50", "51-60", "61+"],
    include_lowest=True
)

#LTI band
df_bi["lti_band"] = pd.cut(
    df_bi["loan_to_income_ratio"],
    bins=[0, 0.1, 0.2, 0.3, 0.4, 0.5, np.inf],
    labels=["0-0.1", "0.1-0.2", "0.2-0.3", "0.3-0.4", "0.4-0.5", "0.5+"],
    include_lowest=True
)

#DTI band
df_bi["dti_band"] = pd.cut(
    df_bi["debt_to_income_ratio"],
    bins=[0, 0.2, 0.4, 0.6, 0.8, np.inf],
    labels=["0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8+"],
    include_lowest=True
)

# Interest rate band
df_bi["int_rate_band"] = pd.cut(
    df_bi["loan_int_rate"],
    bins=[0, 8, 12, 16, 20, np.inf],
    labels=["Under 8%", "8-12%", "12-16%", "16-20%", "Over 20%"],
    include_lowest=True
)

#  Credit history band
df_bi["cred_hist_band"] = pd.cut(
    df_bi["cb_person_cred_hist_length"],
    bins=[0, 3, 7, 15, np.inf],
    labels=["0-3 years", "4-7 years", "8-15 years", "15+ years"],
    include_lowest=True
)

# Credit utilization band
df_bi["util_band"] = pd.cut(
    df_bi["credit_utilization_ratio"],
    bins=[0, 0.3, 0.6, 0.8, np.inf],
    labels=["0-30%", "31-60%", "61-80%", "80%+"],
    include_lowest=True
)

# --- Delinquency level ---
df_bi["delinq_level"] = np.where(
    df_bi["past_delinquencies"] == 0, "Clean History",
    np.where(df_bi["past_delinquencies"] == 1, "Single Late Payment",
    np.where(df_bi["past_delinquencies"] <= 3, "Multiple Issues",
             "Chronic Problems"))
)

# không có NaN trong các band
band_cols = [
    "income_band", "age_band", "lti_band", "dti_band",
    "int_rate_band", "cred_hist_band", "util_band", "delinq_level"
]

# SORT ORDER CHO POWER BI
income_order = {
    "< 30K": 1,
    "30-60K": 2,
    "60-100K": 3,
    "100-200K": 4,
    "200K+": 5
}

age_order = {
    "18-23": 1,
    "24-30": 2,
    "31-36": 3,
    "37-40": 4,
    "41-50": 5,
    "51-60": 6,
    "61+": 7
}

lti_order = {
    "0-0.1": 1,
    "0.1-0.2": 2,
    "0.2-0.3": 3,
    "0.3-0.4": 4,
    "0.4-0.5": 5,
    "0.5+": 6
}

dti_order = {
    "0-0.2": 1,
    "0.2-0.4": 2,
    "0.4-0.6": 3,
    "0.6-0.8": 4,
    "0.8+": 5
}

int_rate_order = {
    "Under 8%": 1,
    "8-12%": 2,
    "12-16%": 3,
    "16-20%": 4,
    "Over 20%": 5
}

cred_hist_order = {
    "0-3 years": 1,
    "4-7 years": 2,
    "8-15 years": 3,
    "15+ years": 4
}

util_order = {
    "0-30%": 1,
    "31-60%": 2,
    "61-80%": 3,
    "80%+": 4
}

delinq_order = {
    "Clean History": 1,
    "Single Late Payment": 2,
    "Multiple Issues": 3,
    "Chronic Problems": 4
}

df_bi["income_band_order"] = df_bi["income_band"].astype(str).map(income_order)
df_bi["age_band_order"] = df_bi["age_band"].astype(str).map(age_order)
df_bi["lti_band_order"] = df_bi["lti_band"].astype(str).map(lti_order)
df_bi["dti_band_order"] = df_bi["dti_band"].astype(str).map(dti_order)
df_bi["int_rate_band_order"] = df_bi["int_rate_band"].astype(str).map(int_rate_order)
df_bi["cred_hist_band_order"] = df_bi["cred_hist_band"].astype(str).map(cred_hist_order)
df_bi["util_band_order"] = df_bi["util_band"].astype(str).map(util_order)
df_bi["delinq_level_order"] = df_bi["delinq_level"].map(delinq_order)

 tạo bảng Power BI

In [8]:

cot_powerbi = [
    # Identity
    "client_ID",

    # Target
    "loan_status",
    "loan_status_label",

    #  Demographics
    "person_age", "age_band", "age_band_order", "person_income", "income_band", "income_band_order", "gender", "marital_status", "education_level",
    "person_home_ownership",

    # Employment
    "person_emp_length", "employment_type",

    # Geography
    "country", "state", "city", "city_latitude","city_longitude",

    #  Loan Info
    "loan_amnt", "loan_intent", "loan_grade" , "loan_int_rate", "int_rate_band", "int_rate_band_order", "loan_term_months", "loan_percent_income",

    #  Risk Ratios
    "loan_to_income_ratio", "lti_band", "lti_band_order", "debt_to_income_ratio", "dti_band", "dti_band_order",

    #  Credit Behavior
    "credit_utilization_ratio", "util_band", "util_band_order", "cb_person_default_on_file", "prior_default_label", "cb_person_cred_hist_length", "cred_hist_band", "cred_hist_band_order",
      "open_accounts", "past_delinquencies",  "delinq_level", "delinq_level_order",
]

cot_powerbi = [c for c in cot_powerbi if c in df_bi.columns]
df_powerbi  = df_bi[cot_powerbi].copy()

ghi về sql sever 

In [ ]:


# Chuyển category 
for col in df_powerbi.select_dtypes(include=["category"]).columns:
    df_powerbi[col] = df_powerbi[col].astype("object")

engine = create_engine(
    "mssql+pyodbc://@localhost/NovaBank_CreditRisk"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&Trusted_Connection=yes"
    "&Encrypt=yes"
    "&TrustServerCertificate=yes",
    fast_executemany=True
)

# Ghi bảng
df_powerbi.to_sql(
    "credit_risk_bi", con=engine,
    schema="dbo", if_exists="replace",
    index=False, chunksize=1000
)

# Verify
with engine.connect() as conn:
    n = conn.execute(
        text("SELECT COUNT(*) FROM dbo.credit_risk_bi")
    ).scalar()